# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the six example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


vix_full_sample_lookahead completed


nasdaq_volatility_success completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

100 run records, 22 counted by the page, 1092 events


total              100.00
completed           38.00
blocked             38.00
failed               0.00
rejected             3.00
completed_share      0.38
blocked_share        0.38
failed_share         0.00
rejected_share       0.03
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

In [4]:
from quantlab.monitoring import (
    grounded_narratives,
    planner_refusals,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **planner_refusals(records),
    }
)

approved              35.000000
rejected               3.000000
approval_rate          0.921053
edited                 4.000000
edit_rate              0.114286
approval_median_s      0.041000
approval_p95_s       101.220700
replays                5.000000
matching               5.000000
replay_match_rate      1.000000
narratives            38.000000
grounded              38.000000
grounded_rate          1.000000
refused               24.000000
refused_share          0.240000
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,168
1,V09,LOOK_AHEAD,APPROVE,25
2,P01,PLANNER_REFUSED,BLOCK,24
3,V08,FREQUENCY_MISMATCH,BLOCK,14
4,V03,INVALID_PARAM,BLOCK,10
5,V02,UNKNOWN_CAPABILITY,BLOCK,6
6,E04,ALIGNMENT_LOSS,WARN,4
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,43
1,flag_regime,C07,34
2,resample,C02,29
3,compute_returns,C03,21
4,rolling_volatility,C06,21
5,forward_return,C08,21
6,compare_groups,C09,21
7,hac_mean_difference,C10,21
8,spread,C05,4
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,20,3404.45,4890.1,296.80,786.60,0.000489,0.000868,0.009777
1,narrator,20,576.45,1578.0,70.25,180.70,0.000093,0.000231,0.001855
2,total,20,3980.90,5958.0,367.05,945.55,0.000582,0.001062,0.011632


planner_calls             20.000000
cache_hits                 4.000000
hit_rate                   0.200000
saved_input_tokens     17504.000000
saved_output_tokens     2985.000000
saved_cost                 0.003243
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,99,869.002020,6250.1900,True
2,Validator,99,0.132626,0.3420,True
3,Executor,38,119.668158,187.5405,True
4,Narrator,38,768.402632,3042.2400,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-18T20:47:04.226Z,20261018-204704-a0005c,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-18T20:47:04.198Z,20261018-204704-c0b3a1,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-18T20:47:04.169Z,20261018-204704-0d3610,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-18T20:13:32.358Z,20261018-201332-9829d7,blocked,Validator,V02,The plan has blocking issues: V02:s2
4,2026-10-18T20:13:32.331Z,20261018-201332-b596c4,blocked,Planner,P01,The plan has blocking issues: P01:plan
